In [1]:
# Make samples.yaml for ichorCNA pipeline
import os
import glob

bam_dir = "/ix1/alee/LO_LAB/General/Lab_Data/20260324_Broad_ULPWGS_TBCRC_AURORA_Rahul/bam_file"
out_yaml = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/ichorCNA_SOP/samples.yaml"

bam_files = sorted(glob.glob(os.path.join(bam_dir, "*.bam")))

with open(out_yaml, "w") as f:
    f.write("samples:\n")
    for bam in bam_files:
        sample_name = os.path.basename(bam).replace(".bam", "")
        f.write(f"  {sample_name}: {bam}\n")

print(f"[INFO] Total BAM files found: {len(bam_files)}")
print(f"[INFO] samples.yaml written to: {out_yaml}")

[INFO] Total BAM files found: 784
[INFO] samples.yaml written to: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/ichorCNA_SOP/samples.yaml


In [5]:
# =========================================================
# Script Title:
# Update Metasheet_Analysis.xlsx from AURORA Clinical File
#
# Purpose:
# Match rows using collaborator_sample_id and fill the following
# columns in Metasheet_Analysis.xlsx from
# AUR_ClinicalData_20260323.NO.DATES.bloodID.xlsx:
#   - Clinical_ID
#   - Clinical_ID_2
#   - ER_Status
#   - PR_Status
#   - HER2_Status
#   - HER2_IHC
#
# Notes:
# - Reads Sheet1 from both Excel files
# - Preserves original Excel formatting using openpyxl
# - Saves output as a new updated Excel file
# - Handles duplicate collaborator_sample_id in the clinical file
#   by taking the first non-empty value per column
# ========================================================= 
from openpyxl import load_workbook
import pandas as pd

# =========================================================
# FILE PATHS
# =========================================================
meta_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
clinical_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260323.NO.DATES.bloodID.xlsx"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.updated.xlsx"
sheet_name = "Sheet1"
unmatched_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/unmatched_collaborator_sample_id.csv"

# =========================================================
# COLUMNS TO TRANSFER
# =========================================================
key_col = "collaborator_sample_id"
cols_to_fill = [
    "Clinical_ID",
    "Clinical_ID_2",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "HER2_IHC"
]

# =========================================================
# READ SOURCE CLINICAL FILE
# =========================================================
clinical_df = pd.read_excel(clinical_file, sheet_name=sheet_name, dtype=str)
clinical_df.columns = clinical_df.columns.str.strip()

# clean relevant columns
needed_cols = [key_col] + cols_to_fill
missing_clinical = [c for c in needed_cols if c not in clinical_df.columns]
if missing_clinical:
    raise ValueError(f"Missing required columns in clinical file: {missing_clinical}")

for col in needed_cols:
    clinical_df[col] = clinical_df[col].astype(str).str.strip()
    clinical_df[col] = clinical_df[col].replace({
        "nan": pd.NA,
        "None": pd.NA,
        "": pd.NA
    })

# =========================================================
# CHECK DUPLICATES IN CLINICAL FILE
# =========================================================
dup_counts = clinical_df[key_col].dropna().value_counts()
dups = dup_counts[dup_counts > 1]

print(f"Duplicate {key_col} in clinical file: {len(dups)}")
if len(dups) > 0:
    print("Showing duplicate keys with transferred columns:")
    display(
        clinical_df[clinical_df[key_col].isin(dups.index)]
        .sort_values(key_col)[needed_cols]
    )

# =========================================================
# BUILD LOOKUP
# keep first occurrence for each collaborator_sample_id
# =========================================================
clinical_lookup = (
    clinical_df[needed_cols]
    .dropna(subset=[key_col])
    .drop_duplicates(subset=[key_col], keep="first")
    .set_index(key_col)
    .to_dict("index")
)

print(f"Unique matched keys available in clinical lookup: {len(clinical_lookup)}")

# =========================================================
# LOAD METASHEET WORKBOOK AND PRESERVE FORMATTING
# =========================================================
wb = load_workbook(meta_file)
ws = wb[sheet_name]

# map header names to column index
headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

missing_meta = [c for c in needed_cols if c not in col_idx]
if missing_meta:
    raise ValueError(f"Missing required columns in metasheet: {missing_meta}")

# =========================================================
# UPDATE METASHEET
# =========================================================
matched_rows = 0
unmatched_ids = []
update_counts = {col: 0 for col in cols_to_fill}

for row in range(2, ws.max_row + 1):
    key_val = ws.cell(row=row, column=col_idx[key_col]).value
    key_val = str(key_val).strip() if key_val is not None else None

    if not key_val:
        continue

    if key_val in clinical_lookup:
        matched_rows += 1
        src = clinical_lookup[key_val]

        for col in cols_to_fill:
            new_val = src.get(col, pd.NA)

            # update only when source value is not empty
            if pd.notna(new_val):
                ws.cell(row=row, column=col_idx[col]).value = new_val
                update_counts[col] += 1
    else:
        unmatched_ids.append(key_val)

# =========================================================
# SAVE UPDATED FILE
# =========================================================
wb.save(output_file)

# =========================================================
# SAVE UNMATCHED IDS
# =========================================================
unique_unmatched = sorted(set(unmatched_ids))

if unique_unmatched:
    unmatched_df = pd.DataFrame({key_col: unique_unmatched})
    unmatched_df.to_csv(unmatched_file, index=False)

    print(f"\nUnmatched {key_col}: {len(unique_unmatched)}")
    print(f"Saved unmatched IDs to: {unmatched_file}")
else:
    print(f"\nNo unmatched {key_col} found.")
# =========================================================
# SUMMARY
# =========================================================
print("\nDone.")
print(f"Matched rows: {matched_rows}")
for col in cols_to_fill:
    print(f"{col} updated: {update_counts[col]}")
print(f"Unmatched {key_col}: {len(set(unmatched_ids))}")
print(f"Saved file: {output_file}")

# optional preview of unmatched IDs
if len(set(unmatched_ids)) > 0:
    print("\nExample unmatched IDs:")
    print(list(sorted(set(unmatched_ids)))[:20])

Duplicate collaborator_sample_id in clinical file: 10
Showing duplicate keys with transferred columns:


,collaborator_sample_id,Clinical_ID,Clinical_ID_2,ER_Status,PR_Status,HER2_Status,HER2_IHC
160,ANFC-NB1,AUP_01_11_01,AUP_01_11_02,<NA>,<NA>,<NA>,<NA>
161,ANFC-NB1,AUP_01_11_01,AUP_01_11_02,<NA>,<NA>,<NA>,<NA>
2888,ANMI-NB7,AUP_588_22_39,AUP_588_22_28,<NA>,<NA>,<NA>,<NA>
2892,ANMI-NB7,AUP_588_22_39,AUP_588_22_40,<NA>,<NA>,<NA>,<NA>
5829,AO2K-NB1,AUP_999_12_01,AUP_999_12_02,<NA>,<NA>,<NA>,<NA>
5830,AO2K-NB1,AUP_999_12_01,AUP_999_12_02,<NA>,<NA>,<NA>,<NA>
5965,APJC-NB1,AUP_999_21_01,AUP_999_21_02,<NA>,<NA>,<NA>,<NA>
5966,APJC-NB1,AUP_999_21_01,AUP_999_21_02,<NA>,<NA>,<NA>,<NA>
5990,APJO-NB1,AUP_999_23_01,AUP_999_23_02,<NA>,<NA>,<NA>,<NA>
5989,APJO-NB1,AUP_999_23_01,AUP_999_23_02,<NA>,<NA>,<NA>,<NA>


Unique matched keys available in clinical lookup: 749

Unmatched collaborator_sample_id: 35
Saved unmatched IDs to: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/unmatched_collaborator_sample_id.csv

Done.
Matched rows: 749
Clinical_ID updated: 749
Clinical_ID_2 updated: 747
ER_Status updated: 0
PR_Status updated: 0
HER2_Status updated: 0
HER2_IHC updated: 0
Unmatched collaborator_sample_id: 35
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.updated.xlsx

Example unmatched IDs:
['AL1Y-NB2', 'AM01-NB6', 'AM9R-NB7', 'AM9R-NB8', 'AMCZ-NB5', 'AME4-NB1', 'AMJW-NB5', 'AMOD-NB2', 'AMOP-NB4', 'AMU9-NB5', 'AMUY-NB5', 'AMUZ-NB5', 'AMVG-NB4', 'AMVU-NB1', 'ANMI-NB5', 'ANYM-NB2', 'AO0U-NB2', 'AOC9-NB2', 'AODK-NB1', 'AOGX-NB2']


In [1]:
from openpyxl import load_workbook
import pandas as pd

# =========================================================
# Script Title:
# Fill Patient_id and Sample_id from collaborator_sample_id
# in Metasheet_Analysis.xlsx
#
# Logic:
# collaborator_sample_id = ANMI-NB8
# Patient_id            = AUP-ANMI
# Sample_id             = AUP-ANMI-NB8
#
# Notes:
# - Reads Sheet1
# - Preserves Excel formatting using openpyxl
# - Saves as a new updated file
# =========================================================

meta_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Patient_SampleID.updated.xlsx"
sheet_name = "Sheet1"

fill_only_blank_cells = False   # True = fill only empty cells

def clean_val(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x in ["", "nan", "None", "NA", "N/A", "null", "NULL"]:
        return None
    return x

def make_patient_id(collab_id):
    collab_id = clean_val(collab_id)
    if not collab_id:
        return None
    
    first_part = collab_id.split("-")[0].strip()
    if not first_part:
        return None
    
    return f"AUP-{first_part}"

def make_sample_id(collab_id):
    collab_id = clean_val(collab_id)
    if not collab_id:
        return None
    return f"AUP-{collab_id}"

# =========================================================
# LOAD WORKBOOK
# =========================================================
wb = load_workbook(meta_file)
ws = wb[sheet_name]

headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

required_cols = ["collaborator_sample_id", "Patient_id", "Sample_id"]
missing = [c for c in required_cols if c not in col_idx]
if missing:
    raise ValueError(f"Missing required columns in metasheet: {missing}")

collab_col = col_idx["collaborator_sample_id"]
patient_col = col_idx["Patient_id"]
sample_col = col_idx["Sample_id"]

# =========================================================
# UPDATE SHEET
# =========================================================
processed_rows = 0
patient_updated = 0
sample_updated = 0
skipped_no_collab = 0

for row in range(2, ws.max_row + 1):
    collab_val = clean_val(ws.cell(row=row, column=collab_col).value)
    
    if not collab_val:
        skipped_no_collab += 1
        continue

    processed_rows += 1

    new_patient_id = make_patient_id(collab_val)
    new_sample_id = make_sample_id(collab_val)

    patient_cell = ws.cell(row=row, column=patient_col)
    sample_cell = ws.cell(row=row, column=sample_col)

    old_patient = clean_val(patient_cell.value)
    old_sample = clean_val(sample_cell.value)

    if fill_only_blank_cells:
        if old_patient is None and new_patient_id is not None:
            patient_cell.value = new_patient_id
            patient_updated += 1
        if old_sample is None and new_sample_id is not None:
            sample_cell.value = new_sample_id
            sample_updated += 1
    else:
        if new_patient_id is not None and old_patient != new_patient_id:
            patient_cell.value = new_patient_id
            patient_updated += 1
        if new_sample_id is not None and old_sample != new_sample_id:
            sample_cell.value = new_sample_id
            sample_updated += 1

# =========================================================
# SAVE
# =========================================================
wb.save(output_file)

# =========================================================
# SUMMARY
# =========================================================
print("Done.")
print("Rows with collaborator_sample_id:", processed_rows)
print("Patient_id updated:", patient_updated)
print("Sample_id updated:", sample_updated)
print("Rows skipped due to missing collaborator_sample_id:", skipped_no_collab)
print("Saved file:", output_file)

Done.
Rows with collaborator_sample_id: 784
Patient_id updated: 784
Sample_id updated: 784
Rows skipped due to missing collaborator_sample_id: 0
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Patient_SampleID.updated.xlsx


In [3]:
from openpyxl import load_workbook
import pandas as pd
import os

# =========================================================
# Script Title:
# Fill Patient_id and Sample_id in Sheet2 from filename
# in Metasheet_Analysis.xlsx
#
# Example:
# filename   = ARWN-NB4.bam
# Patient_id = AUP-ARWN
# Sample_id  = AUP-ARWN-NB4
#
# Notes:
# - Reads Sheet2
# - Preserves Excel formatting using openpyxl
# - Saves as a new updated file
# =========================================================

meta_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.Patient_SampleID.updated.xlsx"
sheet_name = "Sheet2"

fill_only_blank_cells = False   # True = fill only empty cells

def clean_val(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x in ["", "nan", "None", "NA", "N/A", "null", "NULL"]:
        return None
    return x

def strip_extension(filename):
    filename = clean_val(filename)
    if not filename:
        return None
    # removes only the last extension, e.g. .bam
    return os.path.splitext(filename)[0]

def make_patient_id_from_filename(filename):
    base = strip_extension(filename)
    if not base:
        return None

    first_part = base.split("-")[0].strip()
    if not first_part:
        return None

    return f"AUP-{first_part}"

def make_sample_id_from_filename(filename):
    base = strip_extension(filename)
    if not base:
        return None
    return f"AUP-{base}"

# =========================================================
# LOAD WORKBOOK
# =========================================================
wb = load_workbook(meta_file)
ws = wb[sheet_name]

headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

required_cols = ["filename", "Patient_id", "Sample_id"]
missing = [c for c in required_cols if c not in col_idx]
if missing:
    raise ValueError(f"Missing required columns in {sheet_name}: {missing}")

filename_col = col_idx["filename"]
patient_col = col_idx["Patient_id"]
sample_col = col_idx["Sample_id"]

# =========================================================
# UPDATE SHEET
# =========================================================
processed_rows = 0
patient_updated = 0
sample_updated = 0
skipped_no_filename = 0

for row in range(2, ws.max_row + 1):
    filename_val = clean_val(ws.cell(row=row, column=filename_col).value)

    if not filename_val:
        skipped_no_filename += 1
        continue

    processed_rows += 1

    new_patient_id = make_patient_id_from_filename(filename_val)
    new_sample_id = make_sample_id_from_filename(filename_val)

    patient_cell = ws.cell(row=row, column=patient_col)
    sample_cell = ws.cell(row=row, column=sample_col)

    old_patient = clean_val(patient_cell.value)
    old_sample = clean_val(sample_cell.value)

    if fill_only_blank_cells:
        if old_patient is None and new_patient_id is not None:
            patient_cell.value = new_patient_id
            patient_updated += 1
        if old_sample is None and new_sample_id is not None:
            sample_cell.value = new_sample_id
            sample_updated += 1
    else:
        if new_patient_id is not None and old_patient != new_patient_id:
            patient_cell.value = new_patient_id
            patient_updated += 1
        if new_sample_id is not None and old_sample != new_sample_id:
            sample_cell.value = new_sample_id
            sample_updated += 1

# =========================================================
# SAVE
# =========================================================
wb.save(output_file)

# =========================================================
# SUMMARY
# =========================================================
print("Done.")
print("Rows with filename:", processed_rows)
print("Patient_id updated:", patient_updated)
print("Sample_id updated:", sample_updated)
print("Rows skipped due to missing filename:", skipped_no_filename)
print("Saved file:", output_file)

Done.
Rows with filename: 784
Patient_id updated: 784
Sample_id updated: 784
Rows skipped due to missing filename: 0
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.Patient_SampleID.updated.xlsx


In [5]:
from openpyxl import load_workbook
import pandas as pd
import re

# =========================================================
# Script Title:
# Fill Histological_Type in Sheet2 from Clinical_Pathology
# using Patient_id ~ Patient_BCR_Barcode matching
#
# Source:
#   AUP_master_annot_sheet_01-09-2026.xlsx
#   Sheet: Clinical_Pathology
#
# Target:
#   Metasheet_Analysis.Sheet2.Patient_SampleID.updated.xlsx
#   Sheet: Sheet2
#
# Matching logic:
# - normalize Patient_id and Patient_BCR_Barcode
# - uppercase
# - remove spaces
# - remove "AUP-" prefix
# - compare normalized IDs
#
# Output:
# - fills Histological_Type in Sheet2
# - preserves formatting
# - saves as a new Excel file
# =========================================================

source_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUP_master_annot_sheet_01-09-2026.xlsx"
target_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.Patient_SampleID.updated.xlsx"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.Histology.updated.xlsx"

source_sheet = "Clinical_Pathology"
target_sheet = "Sheet2"

target_key_col = "Patient_id"
target_fill_col = "Histological_Type"

source_key_col = "Patient_BCR_Barcode"
source_value_col = "Histological Type"

fill_only_blank_cells = False   # True = fill only empty Histological_Type cells

# =========================================================
# HELPERS
# =========================================================
def clean_val(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x in ["", "nan", "None", "NA", "N/A", "null", "NULL"]:
        return pd.NA
    return x

def normalize_patient_id(x):
    x = clean_val(x)
    if pd.isna(x):
        return pd.NA
    
    x = str(x).upper().strip()
    x = re.sub(r"\s+", "", x)          # remove all spaces
    x = re.sub(r"^AUP-", "", x)        # remove leading AUP-
    x = re.sub(r"[^A-Z0-9]", "", x)    # keep only letters/numbers
    return x if x else pd.NA

def first_nonempty(series):
    for x in series:
        x = clean_val(x)
        if pd.notna(x):
            return x
    return pd.NA

# =========================================================
# READ SOURCE DATA FOR MATCHING / QC
# =========================================================
src_df = pd.read_excel(source_file, sheet_name=source_sheet, dtype=str)
src_df.columns = src_df.columns.str.strip()

required_src = [source_key_col, source_value_col]
missing_src = [c for c in required_src if c not in src_df.columns]
if missing_src:
    raise ValueError(f"Missing required columns in source sheet: {missing_src}")

src_df[source_key_col] = src_df[source_key_col].apply(clean_val)
src_df[source_value_col] = src_df[source_value_col].apply(clean_val)
src_df["normalized_key"] = src_df[source_key_col].apply(normalize_patient_id)

# keep only rows with usable key
src_df_use = src_df.dropna(subset=["normalized_key"]).copy()

# QC: duplicate normalized keys
dup_counts = src_df_use["normalized_key"].value_counts()
dups = dup_counts[dup_counts > 1]

print("Duplicate normalized Patient_BCR_Barcode entries in source:", len(dups))
if len(dups) > 0:
    print("Example duplicate rows:")
    display(
        src_df_use[src_df_use["normalized_key"].isin(dups.index)]
        [[source_key_col, "normalized_key", source_value_col]]
        .sort_values("normalized_key")
        .head(20)
    )

# collapse source by normalized key, taking first non-empty Histological Type
src_collapsed = (
    src_df_use[["normalized_key", source_value_col]]
    .groupby("normalized_key", as_index=False)
    .agg({source_value_col: first_nonempty})
)

print("Usable source rows after collapsing:", src_collapsed.shape[0])
print(f"Non-empty '{source_value_col}' entries:", src_collapsed[source_value_col].notna().sum())

source_lookup = dict(zip(src_collapsed["normalized_key"], src_collapsed[source_value_col]))

# =========================================================
# OPEN TARGET WORKBOOK
# =========================================================
wb = load_workbook(target_file)
ws = wb[target_sheet]

headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

required_tgt = [target_key_col, target_fill_col]
missing_tgt = [c for c in required_tgt if c not in col_idx]
if missing_tgt:
    raise ValueError(f"Missing required columns in target sheet: {missing_tgt}")

target_key_idx = col_idx[target_key_col]
target_fill_idx = col_idx[target_fill_col]

# =========================================================
# UPDATE TARGET SHEET
# =========================================================
processed_rows = 0
matched_rows = 0
updated_rows = 0
unmatched_ids = []

for row in range(2, ws.max_row + 1):
    patient_id = ws.cell(row=row, column=target_key_idx).value
    patient_id = clean_val(patient_id)

    if pd.isna(patient_id):
        continue

    processed_rows += 1
    norm_patient = normalize_patient_id(patient_id)

    if pd.isna(norm_patient):
        unmatched_ids.append(patient_id)
        continue

    new_hist = clean_val(source_lookup.get(norm_patient, pd.NA))

    if pd.notna(new_hist):
        matched_rows += 1
        cell = ws.cell(row=row, column=target_fill_idx)
        old_val = clean_val(cell.value)

        if fill_only_blank_cells:
            if pd.isna(old_val):
                cell.value = new_hist
                updated_rows += 1
        else:
            # safe comparison avoiding pd.NA ambiguity
            old_str = None if pd.isna(old_val) else str(old_val).strip()
            new_str = None if pd.isna(new_hist) else str(new_hist).strip()

            if old_str != new_str:
                cell.value = new_hist
                updated_rows += 1
    else:
        unmatched_ids.append(patient_id)

# =========================================================
# SAVE OUTPUT
# =========================================================
wb.save(output_file)

# =========================================================
# SUMMARY
# =========================================================
print("\nDone.")
print("Rows with Patient_id:", processed_rows)
print("Matched Patient_id to Patient_BCR_Barcode:", matched_rows)
print("Histological_Type updated:", updated_rows)
print("Unmatched Patient_id:", len(set(unmatched_ids)))
print("Saved file:", output_file)

if len(set(unmatched_ids)) > 0:
    print("\nExample unmatched Patient_id:")
    print(list(sorted(set(map(str, unmatched_ids))))[:20])

Duplicate normalized Patient_BCR_Barcode entries in source: 0
Usable source rows after collapsing: 168
Non-empty 'Histological Type' entries: 167

Done.
Rows with Patient_id: 784
Matched Patient_id to Patient_BCR_Barcode: 485
Histological_Type updated: 485
Unmatched Patient_id: 109
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.Histology.updated.xlsx

Example unmatched Patient_id:
['AUP-AL1Y', 'AUP-AL69', 'AUP-AL9C', 'AUP-AL9W', 'AUP-ALA8', 'AUP-ALBN', 'AUP-AM2M', 'AUP-AMEA', 'AUP-AMOD', 'AUP-AMOL', 'AUP-AMT0', 'AUP-AMUA', 'AUP-AMUZ', 'AUP-AMVG', 'AUP-AMVP', 'AUP-AMW2', 'AUP-AMWC', 'AUP-AMZM', 'AUP-ANEY', 'AUP-ANFX']


In [6]:
from openpyxl import load_workbook
import pandas as pd
import re

# =========================================================
# Script Title:
# Fill Histological_Type in Sheet1 from Sheet2
# based on Patient_id matching in Metasheet_Analysis.xlsx
#
# Source sheet: Sheet2
# Target sheet: Sheet1
#
# Notes:
# - preserves formatting using openpyxl
# - normalizes Patient_id before matching
# - saves as a new output file
# =========================================================

excel_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Histology_from_Sheet2.updated.xlsx"

source_sheet = "Sheet2"
target_sheet = "Sheet1"

key_col = "Patient_id"
value_col = "Histological_Type"

fill_only_blank_cells = False   # True = only fill blank Histological_Type in Sheet1

# =========================================================
# HELPERS
# =========================================================
def clean_val(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x in ["", "nan", "None", "NA", "N/A", "null", "NULL"]:
        return pd.NA
    return x

def normalize_patient_id(x):
    x = clean_val(x)
    if pd.isna(x):
        return pd.NA
    x = str(x).upper().strip()
    x = re.sub(r"\s+", "", x)        # remove spaces
    x = re.sub(r"[^A-Z0-9\-]", "", x)
    return x if x else pd.NA

def first_nonempty(series):
    for x in series:
        x = clean_val(x)
        if pd.notna(x):
            return x
    return pd.NA

# =========================================================
# READ SOURCE SHEET FOR MATCHING / QC
# =========================================================
src_df = pd.read_excel(excel_file, sheet_name=source_sheet, dtype=str)
src_df.columns = src_df.columns.str.strip()

required_src = [key_col, value_col]
missing_src = [c for c in required_src if c not in src_df.columns]
if missing_src:
    raise ValueError(f"Missing required columns in {source_sheet}: {missing_src}")

src_df[key_col] = src_df[key_col].apply(clean_val)
src_df[value_col] = src_df[value_col].apply(clean_val)
src_df["normalized_key"] = src_df[key_col].apply(normalize_patient_id)

src_use = src_df.dropna(subset=["normalized_key"]).copy()

# QC: duplicates in source
dup_counts = src_use["normalized_key"].value_counts()
dups = dup_counts[dup_counts > 1]

print("Duplicate normalized Patient_id in source Sheet2:", len(dups))
if len(dups) > 0:
    print("Example duplicate Patient_id rows from Sheet2:")
    display(
        src_use[src_use["normalized_key"].isin(dups.index)]
        [[key_col, "normalized_key", value_col]]
        .sort_values("normalized_key")
        .head(20)
    )

# Collapse source by Patient_id, taking first non-empty Histological_Type
src_collapsed = (
    src_use[["normalized_key", value_col]]
    .groupby("normalized_key", as_index=False)
    .agg({value_col: first_nonempty})
)

print("Usable source rows after collapsing:", src_collapsed.shape[0])
print(f"Non-empty '{value_col}' entries in source:", src_collapsed[value_col].notna().sum())

source_lookup = dict(zip(src_collapsed["normalized_key"], src_collapsed[value_col]))

# =========================================================
# OPEN WORKBOOK
# =========================================================
wb = load_workbook(excel_file)
ws = wb[target_sheet]

headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

required_tgt = [key_col, value_col]
missing_tgt = [c for c in required_tgt if c not in col_idx]
if missing_tgt:
    raise ValueError(f"Missing required columns in {target_sheet}: {missing_tgt}")

target_key_idx = col_idx[key_col]
target_fill_idx = col_idx[value_col]

# =========================================================
# UPDATE TARGET SHEET
# =========================================================
processed_rows = 0
matched_rows = 0
updated_rows = 0
unmatched_ids = []

for row in range(2, ws.max_row + 1):
    patient_id = ws.cell(row=row, column=target_key_idx).value
    patient_id = clean_val(patient_id)

    if pd.isna(patient_id):
        continue

    processed_rows += 1
    norm_patient = normalize_patient_id(patient_id)

    if pd.isna(norm_patient):
        unmatched_ids.append(patient_id)
        continue

    new_hist = clean_val(source_lookup.get(norm_patient, pd.NA))

    if pd.notna(new_hist):
        matched_rows += 1
        cell = ws.cell(row=row, column=target_fill_idx)
        old_val = clean_val(cell.value)

        if fill_only_blank_cells:
            if pd.isna(old_val):
                cell.value = new_hist
                updated_rows += 1
        else:
            old_str = None if pd.isna(old_val) else str(old_val).strip()
            new_str = None if pd.isna(new_hist) else str(new_hist).strip()

            if old_str != new_str:
                cell.value = new_hist
                updated_rows += 1
    else:
        unmatched_ids.append(patient_id)

# =========================================================
# SAVE OUTPUT
# =========================================================
wb.save(output_file)

# =========================================================
# SUMMARY
# =========================================================
print("\nDone.")
print("Rows with Patient_id in Sheet1:", processed_rows)
print("Matched Patient_id from Sheet2:", matched_rows)
print("Histological_Type updated in Sheet1:", updated_rows)
print("Unmatched Patient_id:", len(set(unmatched_ids)))
print("Saved file:", output_file)

if len(set(unmatched_ids)) > 0:
    print("\nExample unmatched Patient_id:")
    print(list(sorted(set(map(str, unmatched_ids))))[:20])

Duplicate normalized Patient_id in source Sheet2: 177
Example duplicate Patient_id rows from Sheet2:


,Patient_id,normalized_key,Histological_Type
783,AUP-AL1Y,AUP-AL1Y,<NA>
778,AUP-AL1Y,AUP-AL1Y,<NA>
779,AUP-AL1Y,AUP-AL1Y,<NA>
782,AUP-AL1Y,AUP-AL1Y,<NA>
781,AUP-AL1Y,AUP-AL1Y,<NA>
780,AUP-AL1Y,AUP-AL1Y,<NA>
776,AUP-AL4L,AUP-AL4L,Ductal
777,AUP-AL4L,AUP-AL4L,Ductal
773,AUP-AL9C,AUP-AL9C,<NA>
772,AUP-AL9C,AUP-AL9C,<NA>


Usable source rows after collapsing: 270
Non-empty 'Histological_Type' entries in source: 161

Done.
Rows with Patient_id in Sheet1: 784
Matched Patient_id from Sheet2: 485
Histological_Type updated in Sheet1: 485
Unmatched Patient_id: 109
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Histology_from_Sheet2.updated.xlsx

Example unmatched Patient_id:
['AUP-AL1Y', 'AUP-AL69', 'AUP-AL9C', 'AUP-AL9W', 'AUP-ALA8', 'AUP-ALBN', 'AUP-AM2M', 'AUP-AMEA', 'AUP-AMOD', 'AUP-AMOL', 'AUP-AMT0', 'AUP-AMUA', 'AUP-AMUZ', 'AUP-AMVG', 'AUP-AMVP', 'AUP-AMW2', 'AUP-AMWC', 'AUP-AMZM', 'AUP-ANEY', 'AUP-ANFX']


In [3]:
from openpyxl import load_workbook
import pandas as pd

# =========================================================
# Fill Sheet2 filename_bam / filename_bai from Sheet3,
# then fill first_MD5_checksum_bam / first_MD5_checksum_bai
# from md5sum.csv in Metasheet_Analysis.xlsx
#
# Logic:
# 1. Read "File name" from Sheet3
# 2. Group filenames by sample stem:
#       ARWN-NB4.bam      -> bam
#       ARWN-NB4.bam.bai  -> bai
# 3. Write paired filenames into Sheet2:
#       filename_bam
#       filename_bai
# 4. Fill MD5 columns from md5sum.csv by exact filename match
#
# Notes:
# - preserves Excel formatting using openpyxl
# - saves as a new file
# - fills Sheet2 row-by-row using sorted paired filenames
# =========================================================

excel_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
csv_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/md5sum.csv"
output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.filename_md5.updated.xlsx"

sheet_source = "Sheet3"
sheet_target = "Sheet2"

source_filename_col = "File name"

target_bam_col = "filename_bam"
target_bam_md5_col = "first_MD5_checksum_bam"
target_bai_col = "filename_bai"
target_bai_md5_col = "first_MD5_checksum_bai"

# True = only fill empty cells in Sheet2
# False = overwrite values in the 4 target columns
fill_only_blank_cells = False

# =========================================================
# HELPERS
# =========================================================
def clean_val(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x in ["", "nan", "None", "NA", "N/A", "null", "NULL"]:
        return pd.NA
    return x

def get_sample_stem(fname):
    """
    Examples:
    ARWN-NB4.bam      -> ARWN-NB4
    ARWN-NB4.bam.bai  -> ARWN-NB4
    """
    fname = clean_val(fname)
    if pd.isna(fname):
        return pd.NA

    fname = str(fname).strip()

    if fname.endswith(".bam.bai"):
        return fname[:-8]   # remove ".bam.bai"
    elif fname.endswith(".bam"):
        return fname[:-4]   # remove ".bam"
    else:
        return pd.NA

# =========================================================
# READ SHEET3 AND BUILD BAM/BAI PAIRS
# =========================================================
sheet3_df = pd.read_excel(excel_file, sheet_name=sheet_source, dtype=str)
sheet3_df.columns = [str(c).strip() for c in sheet3_df.columns]

if source_filename_col not in sheet3_df.columns:
    raise ValueError(f"Missing '{source_filename_col}' in {sheet_source}")

sheet3_df[source_filename_col] = sheet3_df[source_filename_col].apply(clean_val)
sheet3_df = sheet3_df.dropna(subset=[source_filename_col]).copy()

sheet3_df["sample_stem"] = sheet3_df[source_filename_col].apply(get_sample_stem)

usable_sheet3 = sheet3_df.dropna(subset=["sample_stem"]).copy()

paired = {}
for fname in usable_sheet3[source_filename_col]:
    stem = get_sample_stem(fname)
    if pd.isna(stem):
        continue

    if stem not in paired:
        paired[stem] = {"filename_bam": pd.NA, "filename_bai": pd.NA}

    if str(fname).endswith(".bam.bai"):
        paired[stem]["filename_bai"] = fname
    elif str(fname).endswith(".bam"):
        paired[stem]["filename_bam"] = fname

paired_df = pd.DataFrame(
    [
        {
            "sample_stem": stem,
            "filename_bam": vals["filename_bam"],
            "filename_bai": vals["filename_bai"],
        }
        for stem, vals in paired.items()
    ]
).sort_values("sample_stem").reset_index(drop=True)

print("Total usable filenames in Sheet3:", usable_sheet3.shape[0])
print("Total paired rows built from Sheet3:", paired_df.shape[0])
print("Rows with BAM:", paired_df["filename_bam"].notna().sum())
print("Rows with BAI:", paired_df["filename_bai"].notna().sum())

# =========================================================
# READ MD5 CSV
# Expected columns in source CSV:
# filename,md5sum_bam,filename,md5sum_bai
# pandas will typically read them as:
# filename, md5sum_bam, filename.1, md5sum_bai
# =========================================================
md5_df = pd.read_csv(csv_file, dtype=str)
md5_df.columns = [str(c).strip() for c in md5_df.columns]

print("md5sum.csv columns:", md5_df.columns.tolist())

if md5_df.shape[1] != 4:
    raise ValueError(f"Expected 4 columns in md5sum.csv, found {md5_df.shape[1]}")

# rename by position for safety
md5_df.columns = ["bam_filename", "md5sum_bam", "bai_filename", "md5sum_bai"]

for col in md5_df.columns:
    md5_df[col] = md5_df[col].apply(clean_val)

bam_lookup = (
    md5_df[["bam_filename", "md5sum_bam"]]
    .dropna(subset=["bam_filename"])
    .drop_duplicates(subset=["bam_filename"], keep="first")
    .set_index("bam_filename")["md5sum_bam"]
    .to_dict()
)

bai_lookup = (
    md5_df[["bai_filename", "md5sum_bai"]]
    .dropna(subset=["bai_filename"])
    .drop_duplicates(subset=["bai_filename"], keep="first")
    .set_index("bai_filename")["md5sum_bai"]
    .to_dict()
)

print("Unique BAM filenames in md5 CSV:", len(bam_lookup))
print("Unique BAI filenames in md5 CSV:", len(bai_lookup))

# attach md5 values to paired table
paired_df["first_MD5_checksum_bam"] = paired_df["filename_bam"].map(bam_lookup)
paired_df["first_MD5_checksum_bai"] = paired_df["filename_bai"].map(bai_lookup)

print("Paired rows with BAM MD5:", paired_df["first_MD5_checksum_bam"].notna().sum())
print("Paired rows with BAI MD5:", paired_df["first_MD5_checksum_bai"].notna().sum())

# =========================================================
# OPEN WORKBOOK AND TARGET SHEET2
# =========================================================
wb = load_workbook(excel_file)
ws = wb[sheet_target]

headers = [cell.value for cell in ws[1]]
headers_clean = [str(h).strip() if h is not None else None for h in headers]
col_idx = {name: i + 1 for i, name in enumerate(headers_clean)}

required_cols = [
    target_bam_col,
    target_bam_md5_col,
    target_bai_col,
    target_bai_md5_col,
]
missing = [c for c in required_cols if c not in col_idx]
if missing:
    raise ValueError(f"Missing required columns in {sheet_target}: {missing}")

bam_col_idx = col_idx[target_bam_col]
bam_md5_col_idx = col_idx[target_bam_md5_col]
bai_col_idx = col_idx[target_bai_col]
bai_md5_col_idx = col_idx[target_bai_md5_col]

sheet2_data_rows = ws.max_row - 1
paired_rows = paired_df.shape[0]

print("Existing data rows in Sheet2:", sheet2_data_rows)
print("Rows to write from paired Sheet3:", paired_rows)

if paired_rows > sheet2_data_rows:
    raise ValueError(
        f"Sheet2 has only {sheet2_data_rows} data rows, but {paired_rows} paired rows were built from Sheet3."
    )

# =========================================================
# WRITE INTO SHEET2 ROW-BY-ROW
# =========================================================
filename_bam_updated = 0
filename_bai_updated = 0
bam_md5_updated = 0
bai_md5_updated = 0

for i, rec in paired_df.iterrows():
    row = i + 2  # Excel row

    new_bam = clean_val(rec["filename_bam"])
    new_bam_md5 = clean_val(rec["first_MD5_checksum_bam"])
    new_bai = clean_val(rec["filename_bai"])
    new_bai_md5 = clean_val(rec["first_MD5_checksum_bai"])

    # old values
    old_bam = clean_val(ws.cell(row=row, column=bam_col_idx).value)
    old_bam_md5 = clean_val(ws.cell(row=row, column=bam_md5_col_idx).value)
    old_bai = clean_val(ws.cell(row=row, column=bai_col_idx).value)
    old_bai_md5 = clean_val(ws.cell(row=row, column=bai_md5_col_idx).value)

    # filename_bam
    if pd.notna(new_bam):
        if fill_only_blank_cells:
            if pd.isna(old_bam):
                ws.cell(row=row, column=bam_col_idx).value = new_bam
                filename_bam_updated += 1
        else:
            if (pd.isna(old_bam) and pd.notna(new_bam)) or (pd.notna(old_bam) and str(old_bam).strip() != str(new_bam).strip()):
                ws.cell(row=row, column=bam_col_idx).value = new_bam
                filename_bam_updated += 1

    # first_MD5_checksum_bam
    if pd.notna(new_bam_md5):
        if fill_only_blank_cells:
            if pd.isna(old_bam_md5):
                ws.cell(row=row, column=bam_md5_col_idx).value = new_bam_md5
                bam_md5_updated += 1
        else:
            if (pd.isna(old_bam_md5) and pd.notna(new_bam_md5)) or (pd.notna(old_bam_md5) and str(old_bam_md5).strip() != str(new_bam_md5).strip()):
                ws.cell(row=row, column=bam_md5_col_idx).value = new_bam_md5
                bam_md5_updated += 1

    # filename_bai
    if pd.notna(new_bai):
        if fill_only_blank_cells:
            if pd.isna(old_bai):
                ws.cell(row=row, column=bai_col_idx).value = new_bai
                filename_bai_updated += 1
        else:
            if (pd.isna(old_bai) and pd.notna(new_bai)) or (pd.notna(old_bai) and str(old_bai).strip() != str(new_bai).strip()):
                ws.cell(row=row, column=bai_col_idx).value = new_bai
                filename_bai_updated += 1

    # first_MD5_checksum_bai
    if pd.notna(new_bai_md5):
        if fill_only_blank_cells:
            if pd.isna(old_bai_md5):
                ws.cell(row=row, column=bai_md5_col_idx).value = new_bai_md5
                bai_md5_updated += 1
        else:
            if (pd.isna(old_bai_md5) and pd.notna(new_bai_md5)) or (pd.notna(old_bai_md5) and str(old_bai_md5).strip() != str(new_bai_md5).strip()):
                ws.cell(row=row, column=bai_md5_col_idx).value = new_bai_md5
                bai_md5_updated += 1

# =========================================================
# SAVE
# =========================================================
wb.save(output_file)

# =========================================================
# SUMMARY
# =========================================================
print("\nDone.")
print("filename_bam updated:", filename_bam_updated)
print("filename_bai updated:", filename_bai_updated)
print("first_MD5_checksum_bam updated:", bam_md5_updated)
print("first_MD5_checksum_bai updated:", bai_md5_updated)
print("Saved file:", output_file)

Total usable filenames in Sheet3: 1568
Total paired rows built from Sheet3: 784
Rows with BAM: 784
Rows with BAI: 784
md5sum.csv columns: ['filename', 'md5sum_bam', 'filename.1', 'md5sum_bai']
Unique BAM filenames in md5 CSV: 784
Unique BAI filenames in md5 CSV: 784
Paired rows with BAM MD5: 784
Paired rows with BAI MD5: 784
Existing data rows in Sheet2: 784
Rows to write from paired Sheet3: 784

Done.
filename_bam updated: 784
filename_bai updated: 784
first_MD5_checksum_bam updated: 784
first_MD5_checksum_bai updated: 784
Saved file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.Sheet2.filename_md5.updated.xlsx


In [5]:
# Fill "histological_subtype" & "molecular_subtype" information for crossponding samples
import pandas as pd

# =========================
# File paths
# =========================
meta_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"
master_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/07_patient_master_table 1_Daisong.csv"

# =========================
# Read files
# =========================
meta_df = pd.read_excel(meta_file)
master_df = pd.read_csv(master_file)

# Clean column names
meta_df.columns = meta_df.columns.astype(str).str.strip()
master_df.columns = master_df.columns.astype(str).str.strip()

print("Metasheet columns:", meta_df.columns.tolist())
print("Master table columns:", master_df.columns.tolist())

# =========================
# Create patient-level match key
# Example:
# AUP-ANMI-NB8 -> AUP-ANMI
# =========================
meta_df["match_key"] = (
    meta_df["Collaborator_sample_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"-NB\d+$", "", regex=True)
)

master_df["Patient_BCR_Barcode"] = master_df["Patient_BCR_Barcode"].astype(str).str.strip()

# Keep only required mapping columns
master_map = master_df[
    ["Patient_BCR_Barcode", "histological_subtype", "molecular_subtype"]
].drop_duplicates()

# =========================
# Merge metadata
# =========================
merged_df = meta_df.merge(
    master_map,
    left_on="match_key",
    right_on="Patient_BCR_Barcode",
    how="left"
)

# =========================
# Fill target columns in Excel sheet
# =========================
if "Histological_subtype" not in merged_df.columns:
    merged_df["Histological_subtype"] = pd.NA

if "Molecular_subtype" not in merged_df.columns:
    merged_df["Molecular_subtype"] = pd.NA

merged_df["Histological_subtype"] = merged_df["Histological_subtype"].replace("", pd.NA)
merged_df["Molecular_subtype"] = merged_df["Molecular_subtype"].replace("", pd.NA)

merged_df["Histological_subtype"] = merged_df["Histological_subtype"].fillna(merged_df["histological_subtype"])
merged_df["Molecular_subtype"] = merged_df["Molecular_subtype"].fillna(merged_df["molecular_subtype"])

# =========================
# Save output
# =========================
out_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis_filled.xlsx"

merged_df.drop(
    columns=["match_key", "Patient_BCR_Barcode", "histological_subtype", "molecular_subtype"],
    errors="ignore"
).to_excel(out_file, index=False)

print("\nUpdated file saved to:", out_file)

# =========================
# Sanicity check
# =========================
print("\nMatched rows:", merged_df["molecular_subtype"].notna().sum())
print("Unmatched rows:", merged_df["molecular_subtype"].isna().sum())

print("\nExample output:")
print(
    merged_df[
        ["Collaborator_sample_id", "Histological_subtype", "Molecular_subtype"]
    ].head(10)
)
unmatched_df = merged_df[merged_df["molecular_subtype"].isna()][
    ["Collaborator_sample_id", "match_key"]
].drop_duplicates()

print(unmatched_df)

Metasheet columns: ['Broad_entity:sample_id', 'Collaborator_sample_id', 'Clinical_ID', 'Clinical_ID_2', 'Histological_Type', 'Histological_subtype', 'Molecular_subtype', 'Mean_coverage', 'Tumor_Fraction', 'Ploidy', 'Subclone Fraction', 'Frac. Genome Subclonal', 'Frac. CNA Subclonal', 'Frac_Subnucleosomal_90_150', 'Frac_Mononucleosomal_151_250', 'Frac_Dinucleosomal_251_450', 'Frac_Trinucleosomal_451_650']
Master table columns: ['Patient_BCR_Barcode', 'molecular_subtype', 'histological_subtype', 'samples']

Updated file saved to: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis_filled.xlsx

Matched rows: 782
Unmatched rows: 2

Example output:
  Collaborator_sample_id Histological_subtype Molecular_subtype
0           AUP-ANMI-NB8                  NST          HR+HER2-
1           AUP-ANOR-NB2              Unknown          HR+HER2-
2           AUP-ANPW-NB3                  NST          HR+HER2-
3           AUP-ANQU-NB4                  NST          HR+